Exception Handling allows your program to gracefully catch, inspect, and recover from runtime errors(such as invalid user inputs, missing dictionary keys, or failed database connections) instead of crashing the entire server.

## try and Except (catching specific errors)

Place risky operations inside a try block, and handle expected failures inside one or more except blocks.

Always catch specific exception types (like ValueError or KeyError) rather than a bare except:, which accidentally hides bugs and keyboard interrupts

In [2]:
def parse_port(raw_port: str) -> int:
    try:
        port = int(raw_port)
        return port
    except ValueError as err:
        print(f"Invalidport '{raw_port}: {err}. Falling back to 8000")
        return 8000

print(parse_port("8080")) # it will work
print(parse_port("abc"))

8080
Invalidport 'abc: invalid literal for int() with base 10: 'abc'. Falling back to 8000
8000


we can also catch multiple exceptions in a single block using a tuple, or handle each failure type seperately:

In [4]:
def calculate_avg_latency(total_ms: str, request_count: int) ->float:
    try:
        ms = float(total_ms)
        return ms/request_count
    except ValueError:
        print("Error total_ms must be numberc.")
        return 0.0
    except ZeroDivisionError:
        print("Error: rquest_count cannot be zero")
        return 0.0

print(calculate_avg_latency("450.5",0))

Error: rquest_count cannot be zero
0.0


# The Full Flow: try, except , else and finally


Python provides two optional blocks for complete control over exectuion flow:

- else : Runs only if the try block succeeded without raising any execption. Keeping follow-up logic in else prevents accidentally catching errors from code we didn't intend to wrap

- finally: Runs no matter what - whether an execpetion occured, was caught , or even if a return statement was hit. Ideal for closing database sessions or released locks.

In [5]:
def process_payment(raw_amount: str) -> bool:
    print("Opening payment gateway session...")
    try:
        amount = float(raw_amount)
    except ValueError:
        print("Rejected : Non-numerric payment amount.")
        return False
    else:
        #Runs ONLY if the float(raw_amount) succeeded
        print(f"Charging INR {amount:.2f} to customer card...")
        return True
    finally:
        # ALWAYS executes before the function exits
        print("Closing payment gateway session:\n")

process_payment("1499.50")
process_payment("invalid_crash_tester")

Opening payment gateway session...
Charging INR 1499.50 to customer card...
Closing payment gateway session:

Opening payment gateway session...
Rejected : Non-numerric payment amount.
Closing payment gateway session:



False

## Raising Exceptions (raise) & Custom Exceptions

In FastAPI backends, helper funcions and service layers do not return HTTP error string- they raise exceptions when business rules fail , allowing FastAPI's global exception handlers to translate them into clean HTTP status codes(400,401,404)

In [6]:
# Definining a custom exception by inheriting from Python's base Exception class

class UserNotFoundError(Exception):
    """ Raises when a requested user ID does not exist in the database. """
    pass

USERS_DB = {101: "devanath", 102:"alice"}


def get_user_by_id(user_id : int) -> str:
    if user_id not in USERS_DB:
        raise UserNotFoundError(f"User with ID {user_id} was not found.")
    
    return USERS_DB[user_id]


# Calling code catches the custom domain exception

try :
    username = get_user_by_id(99)
    print("Found", username)
except UserNotFoundError as e:
    print(f"404 Not Found -> {e}")

404 Not Found -> User with ID 99 was not found.


# Real - world Backend and API example

Simulating an endpoint payload validator that extracts fields, enforces busines constraints, and guarantee resource cleanup:

In [9]:
class ValidationError(Exception):
    pass

def registerd_user_endpoint(payload: dict[str,str]) -> dict[str, str | int]:
    db_connected = True

    try:

        #1. May raise KeyError if required fields are missing

        username = payload["username"].strip()
        age = int(payload["age"]) #May raise ValueError

        # 2. Enforce custom business rules
        if age < 18:
            raise ValidationError("User must be at least 18 years old")

    except KeyError as missing_field:
        return {"status":400, "error":f"Missing required field : {missing_field}"}
    except ValueError:
        return {"status": 400, "error": "Field 'age' mustbe a valid integer."}
    except ValidationError as err:
        return {"status":422, "error": str(err)}
    else:
        return {"status": 201, "message": f"User '{username} registered successfully"}
    finally:
        db_connected = False #Ensures connection cleanup on every request


print(registerd_user_endpoint({"username": "devanath", "age":"22"}))
print()
print(registerd_user_endpoint({"username": "alez"}))
print()
print(registerd_user_endpoint({"username": "Gifton", "age":"16"}))



{'status': 201, 'message': "User 'devanath registered successfully"}

{'status': 400, 'error': "Missing required field : 'age'"}

{'status': 422, 'error': 'User must be at least 18 years old'}


In [10]:
print(registerd_user_endpoint({"username": "priya", "age": "twenty"}))

{'status': 400, 'error': "Field 'age' mustbe a valid integer."}
